In [1]:
import json
from matplotlib import font_manager
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import pandas as pd
import os
import warnings

warnings.filterwarnings(
    "ignore", category=UserWarning, message="FixedFormatter should only be used together with FixedLocator"
)

In [2]:
def setup_noto_serif():
    noto_path = "./fonts/NotoSerif-Regular.ttf"
    if not os.path.exists(noto_path):
        raise FileNotFoundError("Noto Serif font not found at ./fonts/NotoSerif-Regular.ttf.")
    font_manager.fontManager.addfont(noto_path)
    font_prop = font_manager.FontProperties(fname=noto_path)
    plt.rcParams['font.family'] = font_prop.get_name()
    return font_prop

def load_baseline_f1_scores(scenario, dataset_id):
    suffix = 'ps' if scenario == 'multilabel' else ''
    filename = f'./results/baseline/{dataset_id}{suffix}/automl_autogluon.json'
    with open(filename, 'r') as file:
        data = json.load(file)
    f1_scores = np.round([x['f1_score_weighted'] for x in data['results']], 3)
    best_f1 = np.max(f1_scores)
    mean_time = np.floor(np.mean([x['training_time'] for x in data['results']]))
    return f1_scores, best_f1, mean_time

def load_optimized_f1_scores(dataset_id, baseline_best_f1=None, baseline_mean_time=None, filter_better_than_baseline=True):
    df = pd.read_csv(f'./results/optimized/optuna_openml_{dataset_id}.csv')
    df = df.loc[df['value'] != -1]
    df = df.loc[~df['value'].isna()]
    df = df.loc[df['state'] == 'COMPLETE']
    df = df.loc[(df['oversampling_threshold'] != '0') | (df['undersampling_threshold'] != '0')]

    if filter_better_than_baseline and baseline_best_f1 is not None and baseline_mean_time is not None:
        df = df.loc[
            (df['value'] > baseline_best_f1) |
            ((np.isclose(df['value'], baseline_best_f1)) & (df['total_time'] < baseline_mean_time))
        ]

    times = df['total_time'].to_numpy()
    scores = df['value'].to_numpy()

    return (
        scores,
        round(scores.max(), 3) if len(scores) else float('nan'),
        round(times.mean(), 2) if len(times) else float('nan'),
        round(scores.min(), 3) if len(scores) else float('nan'),
        round(scores.std(), 3) if len(scores) else float('nan'),
        round(times.min(), 2) if len(times) else float('nan'),
        round(times.max(), 2) if len(times) else float('nan'),
        round(times.std(), 2) if len(times) else float('nan'),
    )

In [3]:
font_prop = setup_noto_serif()

In [4]:
datasets = {
    "binary": [
        "31", "37", "44", "1462", "1479", "1510", "40945"
    ],
    "multiclass": [
        "23", "36", "54", "181", "1466", "40691", "40975"
    ],
    "multilabel": [
        "285", "41464", "41465", "41468", "41470", "41471", "41473"
    ]
}

In [5]:
rows = []
for scenario, dataset_list in datasets.items():
    dtype = scenario.capitalize()
    for dsid in dataset_list:
        print(f"Processing Dataset {dsid} ({dtype})")

        try:
            base_scores, base_best_f1, base_time = load_baseline_f1_scores(scenario, dsid)
        except (FileNotFoundError, json.JSONDecodeError) as e:
            print(f"Baseline load failed for {dsid}: {e}")
            base_scores, base_best_f1, base_time = [np.nan], np.nan, np.nan

        try:
            opt_scores, opt_best_f1, opt_mean_time, opt_min_f1, opt_std_f1, opt_min_time, opt_max_time, opt_std_time = \
                load_optimized_f1_scores(dsid, base_best_f1, base_time)
        except (FileNotFoundError, pd.errors.EmptyDataError) as e:
            print(f"Optimized load failed for {dsid}: {e}")
            opt_scores = [np.nan]
            opt_best_f1 = opt_mean_time = opt_min_f1 = opt_std_f1 = opt_min_time = opt_max_time = opt_std_time = np.nan

        delta_f1 = round(opt_best_f1 - base_best_f1, 3) if not np.isnan(opt_best_f1) and not np.isnan(base_best_f1) else np.nan
        delta_time = round(opt_min_time - base_time, 2) if not np.isnan(opt_min_time) and not np.isnan(base_time) else np.nan

        rows.append({
            "Type": dtype,
            "Dataset": dsid,
            "F1_Baseline": base_best_f1,
            "F1_Min": opt_min_f1,
            "F1_Max": opt_best_f1,
            "F1_Mean": round(np.mean(opt_scores), 3) if len(opt_scores) and not np.isnan(opt_scores).all() else np.nan,
            "F1_Std": opt_std_f1,
            "Δ_F1": delta_f1,
            "Time_Baseline": base_time,
            "Time_Min": opt_min_time,
            "Time_Max": opt_max_time,
            "Time_Mean": opt_mean_time,
            "Time_Std": opt_std_time,
            "Δ_Time": delta_time,
        })

results_df = pd.DataFrame(rows)

results_df.to_csv('artifacts/performance_results/performance_results.csv')
results_df.to_excel('artifacts/performance_results/performance_results.xlsx')

results_df

Processing Dataset 31 (Binary)
Processing Dataset 37 (Binary)
Processing Dataset 44 (Binary)
Processing Dataset 1462 (Binary)
Processing Dataset 1479 (Binary)
Processing Dataset 1510 (Binary)
Processing Dataset 40945 (Binary)
Processing Dataset 23 (Multiclass)
Processing Dataset 36 (Multiclass)
Processing Dataset 54 (Multiclass)
Processing Dataset 181 (Multiclass)
Processing Dataset 1466 (Multiclass)
Processing Dataset 40691 (Multiclass)
Processing Dataset 40975 (Multiclass)
Processing Dataset 285 (Multilabel)
Baseline load failed for 285: [Errno 2] No such file or directory: './results/baseline/285ps/automl_autogluon.json'
Processing Dataset 41464 (Multilabel)
Processing Dataset 41465 (Multilabel)
Processing Dataset 41468 (Multilabel)
Processing Dataset 41470 (Multilabel)
Processing Dataset 41471 (Multilabel)
Processing Dataset 41473 (Multilabel)


,Type,Dataset,F1_Baseline,F1_Min,F1_Max,F1_Mean,F1_Std,Δ_F1,Time_Baseline,Time_Min,Time_Max,Time_Mean,Time_Std,Δ_Time
0,Binary,31,0.787,0.788,0.802,0.790,0.003,0.015,10.0,6.16,8.94,7.84,0.55,-3.84
1,Binary,37,0.804,0.804,0.850,0.815,0.011,0.046,22.0,5.71,44.96,13.30,6.12,-16.29
2,Binary,44,0.966,0.966,0.982,0.969,0.003,0.016,80.0,17.04,676.47,121.76,163.39,-62.96
3,Binary,1462,1.000,1.000,1.000,1.000,0.000,0.000,11.0,6.30,10.99,9.43,0.99,-4.70
4,Binary,1479,0.926,0.930,0.934,0.933,0.002,0.008,17.0,20.06,181.36,43.49,23.17,3.06
5,Binary,1510,1.000,1.000,1.000,1.000,0.000,0.000,6.0,5.89,5.89,5.89,0.00,-0.11
6,Binary,40945,0.981,0.981,1.000,1.000,0.002,0.019,8.0,3.84,25.77,4.22,1.27,-4.16
7,Multiclass,23,0.615,0.615,0.647,0.625,0.008,0.032,17.0,11.43,308.67,37.72,55.00,-5.57
8,Multiclass,36,0.991,0.991,0.994,0.992,0.001,0.003,87.0,14.60,25.74,18.41,2.97,-72.40
9,Multiclass,54,0.850,0.850,0.854,0.851,0.002,0.004,17.0,9.18,12.00,10.38,0.68,-7.82


In [6]:
def highlight_delta(val, positive_color, negative_color, zero_color):
    if pd.isna(val):
        return ''
    elif val > 0:
        return f'{positive_color}; color: black'
    elif val < 0:
        return f'{negative_color}; color: black'
    else:
        return f'{zero_color}; color: black'

styled = results_df.style \
    .format({
        "F1_Baseline": "{:.3f}",
        "F1_Min": "{:.3f}",
        "F1_Max": "{:.3f}",
        "F1_Mean": "{:.3f}",
        "F1_Std": "{:.3f}",
        "Δ_F1": "{:+.3f}",
        "Time_Baseline": "{:.2f}",
        "Time_Min": "{:.2f}",
        "Time_Max": "{:.2f}",
        "Time_Mean": "{:.2f}",
        "Time_Std": "{:.2f}",
        "Δ_Time": "{:+.2f}"
    }) \
    .applymap(highlight_delta, subset=["Δ_F1"], positive_color='background-color: #9AFF99', negative_color='background-color: #FFCCC9', zero_color='background-color: #FFFC9E') \
    .applymap(highlight_delta, subset=["Δ_Time"], positive_color='background-color: #FFCCC9', negative_color='background-color: #9AFF99', zero_color='background-color: #FFFC9E') \
    .set_caption("Optimization Results and Comparison with Baseline")

styled.to_html('artifacts/performance_results/performance_results.html')

styled

,Type,Dataset,F1_Baseline,F1_Min,F1_Max,F1_Mean,F1_Std,Δ_F1,Time_Baseline,Time_Min,Time_Max,Time_Mean,Time_Std,Δ_Time
0,Binary,31,0.787,0.788,0.802,0.790,0.003,+0.015,10.00,6.16,8.94,7.84,0.55,-3.84
1,Binary,37,0.804,0.804,0.850,0.815,0.011,+0.046,22.00,5.71,44.96,13.30,6.12,-16.29
2,Binary,44,0.966,0.966,0.982,0.969,0.003,+0.016,80.00,17.04,676.47,121.76,163.39,-62.96
3,Binary,1462,1.000,1.000,1.000,1.000,0.000,+0.000,11.00,6.30,10.99,9.43,0.99,-4.70
4,Binary,1479,0.926,0.930,0.934,0.933,0.002,+0.008,17.00,20.06,181.36,43.49,23.17,+3.06
5,Binary,1510,1.000,1.000,1.000,1.000,0.000,+0.000,6.00,5.89,5.89,5.89,0.00,-0.11
6,Binary,40945,0.981,0.981,1.000,1.000,0.002,+0.019,8.00,3.84,25.77,4.22,1.27,-4.16
7,Multiclass,23,0.615,0.615,0.647,0.625,0.008,+0.032,17.00,11.43,308.67,37.72,55.00,-5.57
8,Multiclass,36,0.991,0.991,0.994,0.992,0.001,+0.003,87.00,14.60,25.74,18.41,2.97,-72.40
9,Multiclass,54,0.850,0.850,0.854,0.851,0.002,+0.004,17.00,9.18,12.00,10.38,0.68,-7.82
